# Evaluate a smooth geometry checkpoint
Run the two setup cells, select a verified checkpoint, and explicitly enable evaluation. Seen-geometry and held-out-family evaluation are separate actions. No training runs here. Historical T1 selections are preserved in a collapsed optional section and are disabled by default.


In [1]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = False
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [2]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    # Only load/install code after repository identity and revision verification.
    import runpy
    _setup_module = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/colab_setup.py"))
    _setup_module["install_checkout"](BOOTSTRAP["project_dir"], extras=("geometry",) if ENABLE_GEOMETRY else ())
if IS_COLAB:
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    print(_reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True))
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local smooth run root:", Path(SETUP["local_output_dir"]) / "G1_smooth_v1/runs")
print("Drive checkpoints:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "checkpoints")
print(SETUP["message"])


Colab: False | imported source: /Users/nicolasackermann/Documents/ChatGPT/PDEep/src/pdeep
Exact Git commit: bcf76427e542c4f612277f1e83265c8874271e23
Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Dependency versions: {
  "pdeep-research": "0.1.0",
  "torch": "2.15.0",
  "numpy": "2.5.4",
  "scipy": "1.18.1",
  "matplotlib": "3.11.2",
  "pytest": "9.2.0",
  "notebook": "7.6.5",
  "nbformat": "5.11.1",
  "nbclient": "0.11.0",
  "ipykernel": "7.4.0",
  "setuptools": "84.2.1"
}
cpu available=True arm
mps available=True MPS float64 unsupported; explicit float32 neural execution only
cuda available=False CUDA unavailable
Local checkpoints: /Users/nicolasackermann/Documents/ChatGPT/PDEep/artifacts/training_runs
Drive checkpoints: None
Local session: no clone, install or Google Drive import/mount; local outputs only.


In [ ]:
SMOOTH_CHECKPOINT = None  # verified local/Drive smooth-update-XXXXXXXX.json
RUN_SMOOTH_EVALUATION = False  # explicit opt-in; default Run All only sets up
SMOOTH_GEOMETRY_HELD_OUT = False  # False: paired seen-geometry cases; True: held-out families/topologies
SMOOTH_DEVICE = "auto"  # portable inference; Colab auto uses validated CUDA
SHOW_REPRESENTATIVE = True


In [ ]:
from pdeep.smooth.workflows import evaluate_checkpoint
from pdeep.smooth.plotting import validation_table, time_plot, representative_field, history_plots
import matplotlib.pyplot as plt
if RUN_SMOOTH_EVALUATION:
    if SMOOTH_CHECKPOINT is None: raise ValueError("Select a verified G1_smooth_v1 checkpoint manifest.")
    effective_device = "cuda" if IS_COLAB and SMOOTH_DEVICE == "auto" else SMOOTH_DEVICE
    if IS_COLAB and not effective_device.startswith("cuda"):
        raise RuntimeError("Colab evaluation requires validated CUDA; no silent fallback.")
    engine, report = evaluate_checkpoint(SMOOTH_CHECKPOINT, SETUP["local_output_dir"],
        geometry_held_out=SMOOTH_GEOMETRY_HELD_OUT, device=effective_device)
    display(Markdown(validation_table(report)))
    print("Checkpoint identity:", report.checkpoint_identity)
    print("Common field MSE:", report.metrics["common"]["field_normalized_MSE"])
    print("Physical weak/boundary/initial diagnostics:", report.architecture_metrics["physical"])
    print("Inference seconds:", report.architecture_metrics["inference_seconds"])
    print("Saved evaluation:", engine.run_dir)
    time_plot(report)
    if SHOW_REPRESENTATIVE and not SMOOTH_GEOMETRY_HELD_OUT:
        representative_field(engine, pde="wave")
    plt.show()
else:
    print("Evaluation pending checkpoint selection and explicit action. Zero optimizer updates.")


The common field table separates PDE, physical quantity and geometry. Time curves aggregate cases; at most one fixed representative wave case is plotted. Weak/boundary/initial diagnostics are valid for the C0/H1 geometry path; global strong second derivatives are not claimed. Tiny checkpoints score their fixed train examples; bounded checkpoints use paired unseen cases on familiar instances. Geometry-held-out scoring uses a separate action and never reads test data.

Exact training resume belongs in02; portable inference here may use a different available device and records that fact. Original T1/D mathematics and metrics below are a separate benchmark.


## Optional historical T1 comparison
The user-selected checkpoint paths and established T1 evaluation code are retained below. Set `ENABLE_T1_EVALUATION=True` only when those files are available. D is the exact nine/eighteen-mode T1 baseline, not a geometry-field posterior.


In [ ]:
ENABLE_T1_EVALUATION = False
CHECKPOINTS = {
    "A": str(ROOT / "artifacts/training_runs/A/A-468a5153d6ff51e3427cf676/inference/artifact-9c59701093d932040cdf018fd2b58d54c3b511e48aa5b07f39ef223fbb56d68e.json"),
    "C": str(ROOT / "artifacts/training_runs/C/C-d09af6be2a2774c846d91cb8/inference/artifact-b96bcfe07727658749732545e3a2ef3ce29b6da47be42d8ed238f9a325456da8.json"),
}  # e.g. {"B0":"/path/to/B0/export.json", "B1":"/path/to/B1/export.json"}
PDE_FAMILIES = ("poisson", "heat", "wave")
DEVICE = "auto"
ALLOW_PARTIAL = False  # explicit opt-in for smoke/bounded/unfinished research runs
CASE_IDS = tuple(range(50)) # (0,)  #  tiny fixed validation subset; not the untouched test split
PLAN_PATH = None  # optional independently saved matched EvaluationPlan JSON
OBSERVATION_DIAGNOSTICS = True  # optional controlled sensor perturbations
SHOW_EXECUTION_CHECKPOINTS = False  # catalog only; never auto-selects models

# Display only: one fixed case/PDE; None selects the lowest evaluated case ID.
REPRESENTATIVE_CASE_IDS = None  # e.g. {"poisson":0, "heat":0, "wave":0}

# Read-only optional display: never executes the benchmark during Notebook Run All.
SHOW_POST_TRAINING_OPTIMIZATION = True
POST_TRAINING_BENCHMARK_PATH = ROOT / "reports/post_training_optimization/result.json"


In [ ]:
def _evaluate_retained_T1():
    from pdeep.notebook_workflows import load_evaluation_selection,checkpoint_catalog
    CONTEXT=initialize(ROOT,device=DEVICE)
    print(CONTEXT.device_message)
    CATALOG=checkpoint_catalog(ROOT,include_partial=SHOW_EXECUTION_CHECKPOINTS)
    print("Available explicit candidates:",CATALOG or "No completed research checkpoint found. Train one in 02_train.ipynb; D is available now.")
    METHODS=load_evaluation_selection(CONTEXT,CHECKPOINTS,allow_partial=ALLOW_PARTIAL)
    for method in METHODS:
        print(method.name,"| status",method.status,"| successful updates",method.training.get("successful_updates",0),"|",method.checkpoint)


    from pdeep.notebook_workflows import make_evaluation_plan,evaluate_selected
    PLAN,EVALUATION_CONFIG=make_evaluation_plan(CONTEXT,pdes=PDE_FAMILIES,case_ids=CASE_IDS,plan_path=PLAN_PATH,observation_diagnostics=OBSERVATION_DIAGNOSTICS)
    print("Plan",PLAN.identity,"| split",PLAN.split,"| cases per PDE",{p:sum(c.pde==p for c in PLAN.cases) for p in PLAN.pdes})
    print("Stationary physical time absent; dynamic times:",PLAN.dynamic_times)


    RESULT=evaluate_selected(CONTEXT,METHODS,PLAN,EVALUATION_CONFIG)
    from pdeep.evaluation_plotting import comparison_table
    display(Markdown(comparison_table(RESULT)))
    print("All quantity/PDE metrics and physical-case aggregation are preserved in JSON.")


    from pdeep.workflow_plotting import plot_evaluation
    from pdeep.evaluation_summary import representative_cases
    PLOTS=plot_evaluation(RESULT,representative_case_ids=REPRESENTATIVE_CASE_IDS)
    print("Representative field/uncertainty case IDs (selected by ID, not error):",
          {p:r["case_id"] for p,r in representative_cases(RESULT["scientific"],REPRESENTATIVE_CASE_IDS).items()})
    from IPython.display import Image
    for path in PLOTS:
        if path.name.startswith("fields_"):display(Image(filename=str(path),width=1000))
    print("All physical cases contribute to tables and time/refinement curves; fields above are representative examples.")


    from pdeep.evaluation_summary import refinement_summary_table
    if any("refinement" in row for rows in RESULT["scientific"]["cases"].values() for row in rows):
        display(Markdown(refinement_summary_table(RESULT)))
    for path in PLOTS:
        if path.name.startswith(("time_","B_refinement")):display(Image(filename=str(path),width=1000))
    print("Time curves aggregate all cases using sqrt(mean MSE). Phi curves show the all-case mean and 10–90% case band.")
    print("B0/B1 have separate weights; a deterministic coefficient estimate is not a posterior sample.")


    from pdeep.evaluation_summary import physics_summary_table
    display(Markdown(physics_summary_table(RESULT)))
    print("Physical residuals aggregate all cases per PDE and equation component; different physical units remain separate.")
    print("Strong residuals check representational consistency, not coefficient recovery. Full per-case/initial/C weak diagnostics remain in JSON.")


    from pdeep.evaluation_summary import coverage_summary_table
    for path in PLOTS:
        if path.name.startswith("D_uncertainty_"):display(Image(filename=str(path),width=1000))
    if "D" in RESULT["scientific"]["cases"]:
        display(Markdown(coverage_summary_table(RESULT)))
    print("D figures: at most one fixed representative case per PDE. Coverage table: all evaluated physical cases.")
    print("Exact D latent variance excludes measurement noise; held-out measurement intervals include noise. Neural posterior uncertainty: unavailable.")
    print("Case-cluster coverage intervals and held-out log predictive densities are retained in the exported JSON.")


    print("Exported JSON, fields and figures:",RESULT["directory"])
    print("Plan:",PLAN.identity,"| scientific result:",RESULT["identity"])
    print("Run metadata:",{name:{"status":r["training_status"],"updates":r["training_selection"]["successful_updates"]} for name,r in RESULT["scientific"]["methods"].items()})
    print("No optimizer updates; no neural uncertainty invented; no test-set selection.")

    # Separate frozen B0+Classical benchmark; validate selected weights/cases/sensors.
    if SHOW_POST_TRAINING_OPTIMIZATION:
        if "B0" not in CHECKPOINTS or not POST_TRAINING_BENCHMARK_PATH.is_file():
            print("Post-training optimization: no compatible saved benchmark selected; no computation launched.")
        else:
            from pdeep.post_training_report import load_for_display,compact_table
            from pdeep.core import ValidationError
            try:
                OPTIMIZATION_BENCHMARK=load_for_display(POST_TRAINING_BENCHMARK_PATH,PLAN,CHECKPOINTS["B0"])
            except ValidationError as error:
                print("Saved post-training benchmark skipped:",error)
            else:
                display(Markdown(f"**Separate frozen B0+Classical optimization benchmark — 50 validation cases/PDE.** Neural timing device: `{OPTIMIZATION_BENCHMARK['device']}`; classical CPU64; {OPTIMIZATION_BENCHMARK['repeats']} warm repeats/case."))
                display(Markdown(compact_table(OPTIMIZATION_BENCHMARK)))
                for filename in ("accuracy_vs_iterations.png","accuracy_vs_wall_clock.png"):
                    path=POST_TRAINING_BENCHMARK_PATH.parent/filename
                    if path.is_file():display(Image(filename=str(path),width=1100))
                print("Warm coefficient/posterior timings include B0 preparation plus each classical tail; D includes complete posterior fit.")
                print("Classical controls and matched D use B0's existing float32 observation copy promoted to CPU64; original-float64 D is a separate control.")
                print("Full budgets, distances, gradients, operation counts, raw timing repeats and shared field-decode cost:",POST_TRAINING_BENCHMARK_PATH.parent/"report.md")


if ENABLE_T1_EVALUATION:
    _evaluate_retained_T1()
else:
    print("Historical T1 selection preserved; evaluation not launched.")


## Phase G — separate geometry comparison
Legacy T1/D results above remain unchanged. Select an A_GEO or C_GEO checkpoint below for aggregated G1 ID, geometry-interpolation and held-out-geometry metrics. This branch uses its saved G1 normalizers and references. D has no exact geometry posterior here; do not compare it as if G1 were the nine-mode T1 benchmark. A few physical plots belong in 01_explore; this evaluation produces a concise table and JSON, not one plot per case.

In [ ]:
GEOMETRY_CHECKPOINT = None  # verified G1 checkpoint .json; does not accept legacy A/C
GEOMETRY_RESULT = None
if GEOMETRY_CHECKPOINT is not None:
    from pdeep.geometric.workflows import evaluate
    GEOMETRY_RESULT = evaluate(GEOMETRY_CHECKPOINT, ROOT / "reports", device="cpu",
        splits=("validation", "interpolation", "geometry_held_out"), indices=(0, 1))
    for row in GEOMETRY_RESULT["summary"]: print(row)
else:
    print("No geometry checkpoint selected: zero G1 evaluation jobs.")